# IBM Applied Data Science Capstone
## Falcon 9 landing prediction - SpaceX API data collection

**Learner:** Djessi Jorge  
**Completed:** 4 August 2026

This notebook documents an original, reproducible pipeline for collecting launch records,
resolving nested SpaceX identifiers and preparing the 90-record Falcon 9 analytical snapshot.
A local course snapshot is used by default so the notebook remains reproducible if the API changes.

### Objectives

1. Request the course SpaceX API snapshot.
2. Resolve rocket, payload, launchpad and core identifiers through the SpaceX v4 API.
3. Normalise the nested JSON into one row per launch.
4. Retain Falcon 9 launches, impute missing payload mass and export `dataset_part_1.csv`.

In [1]:
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import requests

COURSE_API_SNAPSHOT = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)
SPACEX_API = "https://api.spacexdata.com/v4"
LOCAL_SNAPSHOT = Path("dataset_part_1.csv")
RUN_LIVE_API = False

In [2]:
session = requests.Session()

def get_json(url, timeout=30):
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

@lru_cache(maxsize=None)
def resolve(kind, object_id):
    if object_id is None:
        return {}
    return get_json(f"{SPACEX_API}/{kind}/{object_id}")

def collect_falcon9_launches():
    raw_launches = get_json(COURSE_API_SNAPSHOT)
    records = []

    for launch in raw_launches:
        rocket = resolve("rockets", launch.get("rocket"))
        if rocket.get("name") != "Falcon 9":
            continue

        payload_id = (launch.get("payloads") or [None])[0]
        core_link = (launch.get("cores") or [{}])[0]
        payload = resolve("payloads", payload_id)
        pad = resolve("launchpads", launch.get("launchpad"))
        core = resolve("cores", core_link.get("core"))

        outcome = f"{core_link.get('landing_success')} {core_link.get('landing_type')}"
        records.append(
            {
                "FlightNumber": launch.get("flight_number"),
                "Date": pd.to_datetime(launch.get("date_utc")).date(),
                "BoosterVersion": rocket.get("name"),
                "PayloadMass": payload.get("mass_kg"),
                "Orbit": payload.get("orbit"),
                "LaunchSite": pad.get("name"),
                "Outcome": outcome,
                "Flights": core.get("asds_attempts", 0)
                + core.get("rtls_attempts", 0),
                "GridFins": core_link.get("gridfins"),
                "Reused": core_link.get("reused"),
                "Legs": core_link.get("legs"),
                "LandingPad": core_link.get("landpad"),
                "Block": core.get("block"),
                "ReusedCount": core.get("reuse_count"),
                "Serial": core.get("serial"),
                "Longitude": pad.get("longitude"),
                "Latitude": pad.get("latitude"),
            }
        )

    frame = pd.DataFrame.from_records(records)
    frame["FlightNumber"] = range(1, len(frame) + 1)
    frame["PayloadMass"] = frame["PayloadMass"].fillna(
        frame["PayloadMass"].mean()
    )
    return frame

In [3]:
if RUN_LIVE_API:
    launches = collect_falcon9_launches()
    launches.to_csv(LOCAL_SNAPSHOT, index=False)
else:
    launches = pd.read_csv(LOCAL_SNAPSHOT)

print(f"Falcon 9 records: {len(launches)}")
print(f"Columns: {launches.shape[1]}")
launches.head(5)

Falcon 9 records: 90
Columns: 17


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [4]:
quality = pd.DataFrame(
    {
        "dtype": launches.dtypes.astype(str),
        "missing": launches.isna().sum(),
        "unique": launches.nunique(dropna=False),
    }
)
quality

,dtype,missing,unique
FlightNumber,int64,0,90
Date,str,0,90
BoosterVersion,str,0,1
PayloadMass,float64,0,68
Orbit,str,0,11
LaunchSite,str,0,3
Outcome,str,0,8
Flights,int64,0,6
GridFins,bool,0,2
Reused,bool,0,2


In [5]:
assert launches.shape[0] == 90, "The course snapshot should contain 90 Falcon 9 records."
assert launches["PayloadMass"].isna().sum() == 0
assert launches["FlightNumber"].is_unique

summary = launches[["FlightNumber", "PayloadMass", "Flights", "ReusedCount"]].describe().round(2)
summary

,FlightNumber,PayloadMass,Flights,ReusedCount
count,90.00,90.00,90.00,90.00
mean,45.50,6123.55,1.79,3.19
std,26.12,4732.12,1.21,4.19
min,1.00,350.00,1.00,0.00
25%,23.25,2510.75,1.00,0.00
50%,45.50,4701.50,1.00,1.00
75%,67.75,8912.75,2.00,4.00
max,90.00,15600.00,6.00,13.00


### Result

The normalised snapshot contains **90 Falcon 9 launches**. Payload mass is complete after
mean imputation, each launch has a unique flight number, and launch/core/payload metadata is
available for wrangling and predictive modelling.